In [1]:
import polars as pl
from pathlib import Path

pl.set_random_seed(42)

rmsd = pl.read_parquet(
    "../../data/sourced_antibody_v1/no_antibody/assay_scaffold_simple_metrics.parquet"
).filter(pl.col("mean_pae").is_not_null())

rmsd = rmsd.with_columns(
    pl.when(
        pl.col("mean_pae") < 5,
        pl.col("overall_rmsd") <= 2,
        pl.col("epitope_chunk_rmsd_vs_mpnn") <= 1,
        pl.col("af3_clash_resindices").list.len() == 0,
    )
    .then(pl.lit(True))
    .otherwise(pl.lit(False))
    .alias("success")
)

true_dir = Path("../../data/abdb/complex_pdbfiles/cleaned")
pred_dir = Path("../../data/sourced_antibody_v1/no_antibody/af3_predictions")
mpnn_dir = Path("../../data/sourced_antibody_v1/no_antibody/proteinmpnn")

rmsd_filt = rmsd.filter(
    pl.col("mean_pae") < 5,
    pl.col("overall_rmsd") <= 2,
    pl.col("epitope_chunk_rmsd_vs_mpnn") <= 1,
    pl.col("af3_clash_resindices").list.len() == 0,
    # pl.col("mpnn_clash_resindices").list.len() == 0,
)

In [2]:
pl.Config.set_tbl_rows(10)

id_by_success = (
    rmsd.select(
        "id",
        pl.col("scaffolded_epitope_chunk_resindices").list.len().alias("ep_res"),
        "success",
        "epitope_chunks",
    )
    .group_by("id", "ep_res", "epitope_chunks")
    .agg(
        (pl.col("success").sum() / pl.count("success")).alias("prop_success"),
        pl.count("success").alias("shots_on_goal"),
    )
    .with_columns(
        (pl.col("ep_res") / 104).alias("ep_prop"),
    )
    .filter(pl.col("prop_success") > 0)
    .sort(by="prop_success")
)

rows = []
tot = 0

for id in id_by_success.select("id").to_series().to_list():
    focal_rmsd = rmsd_filt.filter(pl.col("id") == id)

    h = focal_rmsd.height

    if h + tot > 403:
        focal_count = (
            focal_rmsd.group_by("contig_id")
            .agg(
                pl.col("contig_string").count().alias("cnt"),
                pl.col("contig_string").first(),
            )
            .sort(by="cnt", descending=False)
        )

        contig = focal_count.select("contig_string").to_series().to_list()
        cnt = focal_count.select("cnt").to_series().to_list()

        captured_contig = []
        for s, c in zip(contig, cnt):
            if c > 10:
                break

            else:
                rows.append(focal_rmsd.filter(pl.col("contig_string") == s))
                captured_contig.append(s)
                tot += c

        rows.append(
            focal_rmsd.join(
                pl.DataFrame({"contig_string": captured_contig}),
                on="contig_string",
                how="anti",
            ).sample(403 - tot)
        )

    else:
        rows.append(focal_rmsd)

        tot += h

keep = pl.concat(rows)

In [43]:
keep.write_parquet(
    "../../data/sourced_antibody_v1/no_antibody/assay_scaffold_simple_metrics_403.parquet"
)

with open(
    "../../data/sourced_antibody_v1/no_antibody/sourced_antibody_v1_no_antibody_simple_403.fasta",
    "w",
) as f:
    for row in keep.iter_rows(named=True):
        f.write(f">{row['scaffolded_epitope_seq']}\n")
        f.write(row["scaffolded_epitope_seq"] + "\n")

In [29]:
pl.Config.set_tbl_rows(-1)

rmsd_filt.filter(pl.col("id") == "7ox3_0P").group_by("contig_id").agg(
    pl.col("contig_string").count().alias("cnt"), pl.col("contig_string").first()
).sort(by="cnt", descending=True)

contig_id,cnt,contig_string
i64,u32,str
47,43,"""16-16/A6-13/51-51/A63-75/16-16"""
5,41,"""17-17/A6-13/49-49/A63-75/17-17"""
49,40,"""14-14/A6-13/56-56/A63-75/13-13"""
30,35,"""15-15/A6-13/53-53/A63-75/15-15"""
16,33,"""20-20/A6-13/44-44/A63-75/19-19"""
10,32,"""16-16/A6-13/52-52/A63-75/15-15"""
46,32,"""19-19/A6-13/46-46/A63-75/18-18"""
24,32,"""15-15/A6-13/54-54/A63-75/14-14"""
18,28,"""18-18/A6-13/47-47/A63-75/18-18"""


In [ ]:
# pl.Config.set_tbl_rows(-1)

# rmsd.select(
#     "id",
#     pl.col("scaffolded_epitope_chunk_resindices").list.len().alias("ep_res"),
#     "success",
#     "epitope_chunks",
# ).group_by("id", "ep_res", "epitope_chunks").agg(
#     (pl.col("success").sum() / pl.count("success")).alias("prop_success"),
#     pl.count("success").alias("shots_on_goal"),
# ).with_columns(
#     (pl.col("ep_res") / 104).alias("ep_prop"),
# ).sort(
#     by="ep_prop", descending=True
# )

id,ep_res,epitope_chunks,prop_success,shots_on_goal,ep_prop
str,u32,i64,f64,u32,f64
"""2xqb_0P""",56,2,0.0,2624,0.538462
"""8t58_0P""",56,2,0.0,322,0.538462
"""3ma9_0P""",54,2,0.0,2368,0.519231
"""8j7e_0P""",46,1,0.0,7,0.442308
"""7zxk_1P""",40,2,0.0,398,0.384615
"""5eu7_1P""",38,2,0.0,3648,0.365385
"""8oxv_0P""",36,2,0.0,418,0.346154
"""6ye3_2P""",36,2,0.0,3392,0.346154
"""5kw9_0P""",36,2,0.0,3392,0.346154


In [12]:
rmsd_filt.select(
    "id",
    pl.col("scaffolded_epitope_chunk_resindices").list.len().alias("ep_res"),
    pl.col("scaffolded_epitope_seq").str.len_chars().alias("total_res"),
).unique().with_columns((pl.col("ep_res") / pl.col("total_res")).alias("ep_prop")).sort(
    by="ep_prop", descending=True
)

id,ep_res,total_res,ep_prop
str,u32,u32,f64
"""4xwo_5P""",31,104,0.298077
"""8pww_0P""",27,104,0.259615
"""8jnk_3P""",26,104,0.25
"""8db4_0P""",22,104,0.211538
"""6xxv_0P""",22,104,0.211538
"""7ox3_0P""",21,104,0.201923
"""6o9i_0P""",19,104,0.182692
"""8cz8_1P""",16,104,0.153846
"""5fhx_1P""",15,104,0.144231


In [14]:
from mdaf3.AF3OutputParser import AF3Output
import MDAnalysis as mda
from MDAnalysis.analysis import rms
from MDAnalysis.analysis.align import alignto
import numpy as np


def num_clashing_interactions_mpnn(row, af3_dir, mpnn_dir, true_dir):
    af3_output = AF3Output(af3_dir / row["assay_scaffolded_epitope_id"])
    af3_u = af3_output.get_mda_universe()
    af3_ris = row["assay_scaffolded_epitope_chunk_resindices"]

    mpnn_u = mda.Universe(
        ((mpnn_dir / row["id"]) / str(row["contig_id"]))
        / (
            row["id"]
            + "_"
            + str(row["rfd_id"])
            + "_fixed_dldesign_"
            + str(row["mpnn_id"])
            + ".pdb"
        )
    )
    mpnn_ris = row["scaffolded_epitope_chunk_resindices"]

    true_u = mda.Universe(true_dir / (row["id"] + ".pdb"))
    true_u.atoms.write("true.pdb")

    true_ris = row["epitope_chunk_resindices"]

    alignto(
        mpnn_u.residues[mpnn_ris].atoms.select_atoms("backbone"),
        true_u.residues[true_ris].atoms.select_atoms("backbone"),
    )

    # print(mpnn_u.residues[mpnn_ris].atoms.select_atoms("name CA").positions[:10])
    # print(true_u.residues[true_ris].atoms.select_atoms("name CA").positions[:10])

    mpnn_combined_u = mda.Merge(
        mpnn_u.atoms, true_u.atoms.select_atoms("segid B or segid C")
    )
    mpnn_combined_u.atoms.write("mpnn_combined.pdb")

    tmp_boolmask = np.full(len(mpnn_u.residues), False)
    tmp_boolmask[mpnn_ris] = True
    unintended_res = mpnn_combined_u.select_atoms("segid A").residues[~tmp_boolmask]

    row["mpnn_clash_resindices"] = list(
        mpnn_combined_u.select_atoms(
            "((segid B or segid C) and (not name H*)) and around 4 group unintended",
            unintended=unintended_res.atoms,
        ).residues.resindices
    )

    alignto(
        af3_u.residues[af3_ris].atoms.select_atoms("backbone"),
        true_u.residues[true_ris].atoms.select_atoms("backbone"),
    )
    # af3_u.atoms.chainIDs = ["D"] * len(af3_u.atoms)
    # af3_combined_u = mda.Merge(
    #     af3_u.atoms, true_u.atoms
    # )

    af3_combined_u = mda.Merge(
        af3_u.atoms, true_u.atoms.select_atoms("segid B or segid C")
    )
    af3_combined_u.atoms.write("af3_combined.pdb")
    tmp_boolmask = np.full(len(af3_u.residues), False)
    tmp_boolmask[af3_ris] = True
    unintended_res = af3_combined_u.select_atoms("segid A").residues[~tmp_boolmask]
    # print(unintended_res)
    # print(unintended_res.atoms.select_atoms("segid A and (not name H*)"))

    l = list(
        af3_combined_u.atoms.select_atoms(
            "((segid B or segid C) and (not name H*)) and around 4 group unintended",
            unintended=unintended_res.atoms,
        ).residues.resindices
    )

    # if len(l) > 0:
    #     print("Clash dectected")

    row["af3_clash_resindices"] = l

    return row

In [6]:
from mdaf3.FeatureExtraction import serial_apply, split_apply_combine
import warnings

warnings.filterwarnings("ignore")


rmsd = split_apply_combine(
    rmsd, num_clashing_interactions_mpnn, pred_dir, mpnn_dir, true_dir, chunksize=15
)

100%|██████████| 62254/62254 [07:13<00:00, 143.72it/s]


In [22]:
no_clash = rmsd.filter(pl.col("af3_clash_resindices").list.len() == 0)

In [23]:
no_clash.filter(
    pl.col("mean_pae") < 5,
    pl.col("overall_rmsd") <= 2,
    pl.col("epitope_chunk_rmsd_vs_mpnn") <= 1,
)

antigen_seq,heavy_seq,light_seq,epitope_seq,id,resolution,light_original_segid,heavy_original_segid,antigen_original_segid,r_work,r_free,antigen_ncbi_taxonomy_id,light_ncbi_taxonomy_id,heavy_ncbi_taxonomy_id,epitope_resindices,epitope_boolmask,contig_string,epitope_chunks,contig_gaps,contig_min,contig_id,contig_length,epitope_chunk_resindices,scaffolded_epitope_chunk_resindices,scaffolded_epitope_resindices,rfd_id,mpnn_id,scaffolded_epitope_seq,assay_scaffolded_epitope_seq,assay_scaffolded_epitope_id,assay_scaffolded_epitope_chunk_resindices,assay_scaffolded_epitope_resindices,overall_rmsd,epitope_chunk_rmsd_vs_mpnn,mean_pae,mpnn_clash_resindices,af3_clash_resindices
str,str,str,str,str,f64,str,str,str,f64,f64,i64,i64,i64,list[i64],list[bool],str,i64,list[i64],list[i64],i64,str,list[i64],list[i64],list[i64],i64,i64,str,str,str,list[i64],list[i64],f64,f64,f64,list[i64],list[i64]
"""DLTVEPNLHSLITSTTHKWIFVGGKGGVGK…","""VQLVESGGGLVQPGGSLRLSCAASGFNLYY…","""DIQMTQSPSSLSASVGDRVTITCRASQSVS…","""FLYERIQIVNQELYEDF""","""4xwo_5P""",2.75,"""R""","""Q""","""N""",0.1956,0.2472,559292,9606,9606,"[196, 199, … 256]","[false, false, … false]","""8-8/A197-218/58-58/A249-257/7-…",2,"[8, 58, 7]","[0, 10, 0]",1,"""104-104""","[196, 197, … 256]","[8, 9, … 96]","[8, 11, … 96]",3,7,"""SEKLKNLVFLSLYETERLIQELISYDMDVN…","""GASEKLKNLVFLSLYETERLIQELISYDMD…","""52bda76cbf653041399f41630c597a…","[10, 11, … 98]","[10, 13, … 98]",0.7155,0.593783,4.943669,[],[]
"""DLTVEPNLHSLITSTTHKWIFVGGKGGVGK…","""VQLVESGGGLVQPGGSLRLSCAASGFNLYY…","""DIQMTQSPSSLSASVGDRVTITCRASQSVS…","""FLYERIQIVNQELYEDF""","""4xwo_5P""",2.75,"""R""","""Q""","""N""",0.1956,0.2472,559292,9606,9606,"[196, 199, … 256]","[false, false, … false]","""21-21/A197-218/31-31/A249-257/…",2,"[21, 31, 21]","[0, 10, 0]",11,"""104-104""","[196, 197, … 256]","[21, 22, … 82]","[21, 24, … 82]",2,1,"""MGTLIVEVEVPEDLEEAEKRFFLSLYETER…","""GAMGTLIVEVEVPEDLEEAEKRFFLSLYET…","""fa49bcaa91e39a9ba6e2de5f545347…","[23, 24, … 84]","[23, 26, … 84]",1.067625,0.594184,4.568315,[],[]
"""DLTVEPNLHSLITSTTHKWIFVGGKGGVGK…","""VQLVESGGGLVQPGGSLRLSCAASGFNLYY…","""DIQMTQSPSSLSASVGDRVTITCRASQSVS…","""FLYERIQIVNQELYEDF""","""4xwo_5P""",2.75,"""R""","""Q""","""N""",0.1956,0.2472,559292,9606,9606,"[196, 199, … 256]","[false, false, … false]","""5-5/A197-218/63-63/A249-257/5-…",2,"[5, 63, 5]","[0, 10, 0]",14,"""104-104""","[196, 197, … 256]","[5, 6, … 98]","[5, 8, … 98]",5,4,"""SEELNFLSLYETERLIQELISYDMDVNTIV…","""GASEELNFLSLYETERLIQELISYDMDVNT…","""aaf74b3d215f6d24a66800606d8e3c…","[7, 8, … 100]","[7, 10, … 100]",1.211804,0.620923,4.877741,[],[]
"""DLTVEPNLHSLITSTTHKWIFVGGKGGVGK…","""VQLVESGGGLVQPGGSLRLSCAASGFNLYY…","""DIQMTQSPSSLSASVGDRVTITCRASQSVS…","""FLYERIQIVNQELYEDF""","""4xwo_5P""",2.75,"""R""","""Q""","""N""",0.1956,0.2472,559292,9606,9606,"[196, 199, … 256]","[false, false, … false]","""5-5/A197-218/63-63/A249-257/5-…",2,"[5, 63, 5]","[0, 10, 0]",14,"""104-104""","[196, 197, … 256]","[5, 6, … 98]","[5, 8, … 98]",5,7,"""SEELNFLSLYETERLIQELISYDMDVNKLV…","""GASEELNFLSLYETERLIQELISYDMDVNK…","""92713251810ba2f9401e1a6e0a6cd9…","[7, 8, … 100]","[7, 10, … 100]",1.041326,0.602041,4.765648,[],[]
"""DLTVEPNLHSLITSTTHKWIFVGGKGGVGK…","""VQLVESGGGLVQPGGSLRLSCAASGFNLYY…","""DIQMTQSPSSLSASVGDRVTITCRASQSVS…","""FLYERIQIVNQELYEDF""","""4xwo_5P""",2.75,"""R""","""Q""","""N""",0.1956,0.2472,559292,9606,9606,"[196, 199, … 256]","[false, false, … false]","""7-7/A197-218/60-60/A249-257/6-…",2,"[7, 60, 6]","[0, 10, 0]",18,"""104-104""","[196, 197, … 256]","[7, 8, … 97]","[7, 10, … 97]",5,6,"""SKADELIFLSLYETERLIQELISYDMDVNT…","""GASKADELIFLSLYETERLIQELISYDMDV…","""bfdbd02afab19251b8a330292f2ff4…","[9, 10, … 99]","[9, 12, … 99]",0.793993,0.538666,4.313558,[],[]
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""NIANSIDILQEKEGHLDFVIIPHYTFLDYY…","""QVQLQESGPGLVKPSETLSLTCTVSGGPIS…","""DIQMTQSPSSLSASVGDRVTITCQASQDIN…","""AFKIAKMDKNYTNE""","""8pww_0P""",1.95,"""B""","""B""","""A""",0.2301,0.2515,36329,9606,9606,"[49, 53, … 136]","[fa

In [18]:
no_clash.filter(
    pl.col("mean_pae") < 5,
    pl.col("overall_rmsd") <= 2,
    pl.col("epitope_chunk_rmsd_vs_mpnn") <= 1,
).select("id").unique()

id
str
"""8pww_0P"""
"""8jnk_3P"""
"""7ox3_0P"""
"""8db4_0P"""
"""4xwo_5P"""
"""8cz8_1P"""
"""5fhx_1P"""


In [ ]:
rmsd.filter(
    pl.col("af3_clash_resindices").list.len() == 0,
    pl.col("mpnn_clash_resindices").list.len() != 0,
).filter(
    pl.col("mean_pae") < 5,
    pl.col("overall_rmsd") <= 2,
    pl.col("epitope_chunk_rmsd_vs_mpnn") <= 1,
).select(
    pl.col("mpnn_clash_resindices").list.len().max()
)

mpnn_clash_resindices
u32
2


In [15]:
rmsd = split_apply_combine(
    rmsd.filter(
        pl.col("af3_clash_resindices").list.len() == 0,
        pl.col("mpnn_clash_resindices").list.len() != 0,
    ).filter(
        pl.col("mean_pae") < 5,
        pl.col("overall_rmsd") <= 2,
        pl.col("epitope_chunk_rmsd_vs_mpnn") <= 1,
    )[
        0
    ],
    num_clashing_interactions_mpnn,
    pred_dir,
    mpnn_dir,
    true_dir,
    chunksize=15,
)

100%|██████████| 1/1 [00:00<00:00,  3.54it/s]
